# DiCo-NLI Track 1 - best config, then error analysis

This notebook does one job: **train the configuration that won the sweep, then show exactly what it gets wrong.**

**Best config** (from 6 runs, 2 seeds, freeze 0 / 6 / 9):

| | best-epoch F1 | last-epoch F1 | soft | hard | trainable |
|---|---|---|---|---|---|
| freeze 0 | 0.8356 | 0.8065 | 0.9206 | 0.8448 | 86.0M (30.9%) |
| freeze 6 | 0.8206 | 0.8008 | 0.9242 | 0.8394 | 43.5M (15.6%) |
| **freeze 9** | **0.8359** | **0.8286** | **0.9531** | **0.8881** | **22.3M (8.0%)** |

Backbone `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, embeddings frozen, reversal-equivariant
inference, and **no structural prior**: `TWIN_PRIOR = "off"` is the default. The `pair_id`
exploit is still implemented and measurable in section 6, but it is not part of the recipe.

**Caveat on the numbers below** (measured afterwards): the sweep ran with `PATIENCE = 3`, and
freeze 9 / seed 43 stopped at epoch 7 with 0.8378 - one epoch before its best. Allowed to run
on, the same recipe scored **0.8456 at epoch 8**. The notebook now trains a fixed 12 epochs
with no early stop and exports the best epoch, so the freeze ranking above is a lower bound.

**Runtime**: 2 seeds x ~6 min = ~12 min on a Kaggle GPU. Nothing is retrained if the run
already exists in `RESULTS_DIR`; the grid is resumable.

**What section 5 prints**: a per-label table (support, precision, recall, F1, and each
class's share of the weighted score), the NEG block measured rather than assumed, the
error-pattern table, and the rows behind the largest pattern. Every error is also written
to `failures_epoch<N>.csv` next to the run.


## 1. Setup

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # plain-python fallback (nbconvert / script)
    def display(obj):
        print(obj)


GITHUB_REPO = "https://github.com/AmnO-O/DICO-NLI"
GITHUB_BRANCH = "main"

# Kaggle paths (only active when running on Kaggle's Linux environment).
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/ieltsmater/dico-nli/DiCO-NLI")
IS_KAGGLE = sys.platform.startswith("linux") and KAGGLE_WORKING.is_dir()

# Set to a path to force a specific data directory, or leave None to auto-detect.
DATA_ROOT_OVERRIDE = None


def is_repo_root(path: Path) -> bool:
    return (path / "starter_kit" / "main.py").exists() and (path / "evaluation_functions").is_dir()


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if is_repo_root(candidate):
            return candidate
    return None


def ensure_repo() -> Path:
    # 1) a checkout is already available (local machine, or bundled in the Kaggle dataset)
    candidates = [Path.cwd().resolve()]
    if IS_KAGGLE:
        candidates += [KAGGLE_DATA_ROOT, KAGGLE_WORKING / "DiCO-NLI"]
    for candidate in candidates:
        if candidate.exists():
            root = find_repo_root(candidate)
            if root is not None:
                return root
    # 2) otherwise clone it (Kaggle: enable Internet in the notebook settings)
    clone_dir = (KAGGLE_WORKING if IS_KAGGLE else Path.cwd()) / "DiCO-NLI"
    if not is_repo_root(clone_dir):
        print(f"Cloning {GITHUB_REPO} -> {clone_dir}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(clone_dir)],
            check=True,
        )
    return clone_dir


def resolve_data_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        return Path(DATA_ROOT_OVERRIDE)
    if IS_KAGGLE and (KAGGLE_DATA_ROOT / "train").is_dir() and (KAGGLE_DATA_ROOT / "dev").is_dir():
        return KAGGLE_DATA_ROOT
    return REPO_ROOT / "final_data"


REPO_ROOT = ensure_repo()
STARTER_KIT = REPO_ROOT / "starter_kit"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_functions import score_files  # noqa: E402

DATA_ROOT = resolve_data_root()

def repo_version(root: Path) -> str:
    # Which commit is actually being executed? On Kaggle ensure_repo() can pick
    # the bundled dataset copy instead of a fresh clone, so this matters.
    try:
        out = subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, encoding="utf-8", errors="replace")
        return out.stdout.strip() or "unknown (not a git checkout)"
    except Exception:
        return "unknown"


print("Python  :", sys.executable)
print("Kaggle  :", IS_KAGGLE)
print("Repo    :", REPO_ROOT, "| git", repo_version(REPO_ROOT))
print("GitHub  :", GITHUB_REPO)
print("Data    :", DATA_ROOT)
print("Train   :", (DATA_ROOT / "train").exists())
print("Dev     :", (DATA_ROOT / "dev").exists())


def run_logged_command(cmd, cwd, log_path,
                       echo=("Epoch", "average loss", "Training history", "Trainable parameters",
                             "F1-weighted", "Early stopping")):
    # Run a command, tee its stdout/stderr into log_path, and echo the lines that
    # matter (freeze confirmation, per-epoch loss, per-epoch dev F1) live so you can
    # watch progress. The dev set IS evaluated after every epoch; without
    # "F1-weighted" in this tuple those lines would only land in the log file.
    with Path(log_path).open("w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n\n")
        log.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if any(token in line for token in echo):
                print("    " + line.rstrip())
        returncode = proc.wait()
    return returncode


## 2. The configuration (fixed, not a grid)

In [ ]:
# ----------------------------- chosen configuration -----------------------
# Sweep result: freeze 9 ties freeze 0 on best-epoch F1 and beats it on
# last-epoch F1, soft consistency, hard consistency, seed spread and cost.
MODEL_KEY = "mDeBERTa-v3-base-mnli-xnli"
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
MODEL_LAYERS = 12

TRACK = 1
SEEDS = [42, 43]

# ---- training data scope -------------------------------------------------
# "single" = track 1 only, the honest baseline.
# "mono"   = tracks 1-3: en-en + es-es + eu-eu, 9,126 rows over the same
#            1,760 decisions. Tests whether multilingual regularisation helps.
# "all"    = tracks 1-4, 27,378 rows: adds the six cross-lingual directions.
# Build the two extra files with `python scripts/build_multitrack_train.py`.
TRAIN_SCOPE = "single"
MULTITRACK_FILES = {
    "mono": "dico_nli_train_multilingual.csv",
    "all": "dico_nli_train_multitrack.csv",
}

FREEZE_BOTTOM_LAYERS = 9   # train the top 3 encoder layers + head (22.3M / 8.0%)
FREEZE_EMBEDDINGS = True   # the 193M multilingual embedding table is 69% of the model
MAX_LENGTH = 64
# No early stop: f9/seed43 peaked at epoch 8, but the patience=3 run stopped
# at epoch 7 with 0.8378 and never evaluated epoch 8 (0.8456). The ZIP already
# exports the best epoch, so extra epochs only cost time, not accuracy.
NUM_EPOCHS = 12
PATIENCE = None
LR = 3e-5
BATCH_SIZE = 32
WD = 1e-5
WARMUP_PCT = 0.01
DROPOUT = 0.1
FAST = True
SAVE_PROBS = True   # dump probabilities_dev_*.npy so seeds can be averaged later

# ------------------------- inference-time settings ------------------------
SYMMETRIC = True
SYMMETRIC_MODE = "prob"

# Structural prior from pair_id duplication. This reads how the file was
# built, not what the text means, so it is OFF by default.
# Measured on dev (f9/seed43, epoch 8): off = 0.8456. "full" is worth about
# +0.07 by overriding 106 rows, and only holds if the hidden split duplicates
# pairs the same way the dev split happens to.
#   off    -> honest baseline (default)
#   forbid -> paired rows cannot be NEG  (follows from the label definition)
#   force  -> lone rows must be NEG      (an assumption about the file)
#   full   -> both, the aggressive option
TWIN_PRIOR = "off"

# ----------------------------- output locations ---------------------------
OUTPUT_ROOT = KAGGLE_WORKING if IS_KAGGLE else REPO_ROOT
RESULTS_DIR = OUTPUT_ROOT / "results" / "modernbert_track1_frozen"
LOG_DIR = RESULTS_DIR / "logs"
SUMMARY_CSV = RESULTS_DIR / "frozen_scores.csv"
OVERWRITE = False      # True -> delete and retrain the runs below. Skips are keyed on
                       # the run name alone, so a changed NUM_EPOCHS/PATIENCE/LR will
                       # reuse the old directory unless this is True.
SAVE_MODELS = False    # True -> keep each fine-tuned checkpoint (~1.1 GB per run)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# starter_kit/main.py rejects these values with an AssertionError; fail here
# instead of after a 600 MB model download.
ALLOWED_LR = [1e-6, 5e-6, 8e-6, 9e-6, 1e-5, 1.5e-5, 2e-5, 2.5e-5, 3e-5, 4e-5, 5e-5, 8e-5, 1e-4]
ALLOWED_WD = [1e-6, 5e-6, 8e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
assert 1 <= NUM_EPOCHS <= 100, f"NUM_EPOCHS={NUM_EPOCHS} must be 1..100"
assert 8 <= BATCH_SIZE <= 64 and BATCH_SIZE % 8 == 0, f"BATCH_SIZE={BATCH_SIZE} must be 8..64, multiple of 8"
assert LR in ALLOWED_LR, f"LR={LR} must be one of {ALLOWED_LR}"
assert WD in ALLOWED_WD, f"WD={WD} must be one of {ALLOWED_WD}"
assert 0 <= DROPOUT <= 0.5, f"DROPOUT={DROPOUT} must be 0..0.5"
assert 0 <= WARMUP_PCT <= 0.1, f"WARMUP_PCT={WARMUP_PCT} must be 0..0.1"
assert PATIENCE is None or PATIENCE >= 1, f"PATIENCE={PATIENCE} must be >= 1 or None"
assert SYMMETRIC_MODE in ("prob", "logprob", "logit"), f"SYMMETRIC_MODE={SYMMETRIC_MODE}"
assert TWIN_PRIOR in ("off", "forbid", "force", "full"), f"TWIN_PRIOR={TWIN_PRIOR}"
assert TRAIN_SCOPE in ("single", *MULTITRACK_FILES), f"TRAIN_SCOPE={TRAIN_SCOPE}"
assert 0 <= FREEZE_BOTTOM_LAYERS <= MODEL_LAYERS, "FREEZE_BOTTOM_LAYERS exceeds the model depth"

TOTAL = len(SEEDS)
print(f"Configured {TOTAL} run(s) for track {TRACK} -> {RESULTS_DIR}")
print(f"  {MODEL_KEY}: freeze={FREEZE_BOTTOM_LAYERS}/{MODEL_LAYERS} -> "
      f"top {MODEL_LAYERS - FREEZE_BOTTOM_LAYERS} layer(s) + head"
      f"{' (embeddings frozen)' if FREEZE_EMBEDDINGS else ''}")
print(f"  symmetric={SYMMETRIC} ({SYMMETRIC_MODE}), twin_prior={TWIN_PRIOR}")
print(f"  train_scope={TRAIN_SCOPE}")


## 3. Train the two seeds

In [ ]:
def track_paths(track: int):
    train = DATA_ROOT / "train" / f"dico_nli_train_track{track}_participant_labeled.csv"
    dev = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_participant_labeled.csv"
    ref = DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_reference.csv"
    for path in (train, dev, ref):
        if not path.exists():
            raise FileNotFoundError(path)
    return train, dev, ref


def train_path() -> Path:
    """The CSV this run fits: track 1 alone, or a pre-built combined file."""
    if TRAIN_SCOPE == "single":
        return track_paths(TRACK)[0]
    path = DATA_ROOT / "train" / MULTITRACK_FILES[TRAIN_SCOPE]
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing - run `python scripts/build_multitrack_train.py`"
        )
    return path


def run_key(model_key: str, freeze: int, seed: int) -> str:
    slug = model_key.lower().replace(".", "")
    # Symmetric runs get their own directory: without the suffix they would be
    # skipped as "already scored" by the non-symmetric run of the same config.
    suffix = "_sym" if SYMMETRIC else ""
    if TRAIN_SCOPE != "single":
        suffix += f"_mt-{TRAIN_SCOPE}"
    if TWIN_PRIOR != "off":
        suffix += f"_prior-{TWIN_PRIOR}"
    return f"{slug}_track{TRACK}_f{freeze}_seed{seed}{suffix}"


def build_command(model_id, seed, results_dir, freeze, *, save_path=None):
    train = train_path()
    _, dev, ref = track_paths(TRACK)
    return [
        sys.executable, "-u", "main.py",
        "--model_path", model_id,
        "--tokenizer_path", model_id,
        "--do_train",
        "--train_path", str(train),
        "--do_dev",
        "--dev_path", str(dev),
        "--no-do_test",
        "--seed", str(seed),
        "--dropout", str(DROPOUT),
        "--warmup_pcrt", str(WARMUP_PCT),
        "--lr", str(LR),
        "--batch_size", str(BATCH_SIZE),
        "--wd", str(WD),
        "--num_epochs", str(NUM_EPOCHS),
        *(["--patience", str(PATIENCE)] if PATIENCE is not None else []),
        *(["--fast"] if FAST else []),
        *(["--save_probs"] if SAVE_PROBS else []),
        *(["--symmetric", "--symmetric_mode", SYMMETRIC_MODE] if SYMMETRIC else []),
        *(["--twin_prior", TWIN_PRIOR] if TWIN_PRIOR != "off" else []),
        "--no-is_optuna_trial",
        "--max_length", str(MAX_LENGTH),
        "--results_dir", str(results_dir),
        "--reference_path", str(ref),
        "--freeze_bottom_layers", str(freeze),
        *(["--freeze_embeddings"] if FREEZE_EMBEDDINGS else []),
        *(["--save_path", str(save_path)] if save_path is not None else []),
    ]


TRAINABLE_RE = re.compile(r"Trainable parameters:\s*([\d,]+)\s*/\s*([\d,]+)\s*\(([\d.]+)%\)")


def parse_trainable(log_path: Path):
    # Read the 'Trainable parameters: A / B (P%)' line printed by model.py.
    if not log_path.exists():
        return None, None
    text = log_path.read_text(encoding="utf-8", errors="replace")
    match = TRAINABLE_RE.search(text)
    if match is None:
        return None, None
    trainable = int(match.group(1).replace(",", ""))
    total = int(match.group(2).replace(",", ""))
    return trainable, round(100.0 * trainable / total, 2)


def score_predictions(key, model_key, model_id, freeze, seed, pred_path, ref_path, run_dir, elapsed=None):
    report = score_files(ref_path, pred_path, output_dir=run_dir / "official_scores" / pred_path.stem)
    trainable, trainable_pct = parse_trainable(LOG_DIR / f"{key}.log")
    return {
        "run": key,
        "model": model_key,
        "model_id": model_id,
        "track": TRACK,
        "train_scope": TRAIN_SCOPE,
        "freeze_layers": freeze,
        "freeze_embeddings": FREEZE_EMBEDDINGS,
        "symmetric": SYMMETRIC,
        "symmetric_mode": SYMMETRIC_MODE,
        "twin_prior": TWIN_PRIOR,
        "seed": seed,
        "weighted_f1": report.weighted_f1,
        "soft_cons": report.soft_cons,
        "hard_cons": report.hard_cons,
        "macro_f1": report.classification.macro_f1,
        "accuracy": report.classification.accuracy,
        "instances": report.classification.total_instances,
        "trainable_params": trainable,
        "trainable_pct": trainable_pct,
        "predictions": str(pred_path),
        "minutes": None if elapsed is None else round(elapsed / 60, 2),
    }


def read_history(run_dir: Path):
    """Per-epoch records the trainer rewrites after every epoch (dev F1 included)."""
    path = run_dir / "training_history.json"
    if not path.exists():
        return []
    try:
        history = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return []
    return [row for row in history if "dev_f1_weighted" in row]


def best_epoch_of(history):
    """1-based epoch with the highest dev F1, or None when unavailable."""
    if not history:
        return None
    return int(max(history, key=lambda row: row["dev_f1_weighted"])["epoch"])


def score_run(key, model_key, model_id, freeze, seed, run_dir, elapsed=None):
    """Score the run at its **last** epoch and at its **best dev** epoch.

    The trainer writes one ``predictions_dev_*.csv`` per epoch, so the last one is
    not necessarily the best. ``weighted_f1`` / ``soft_cons`` / ``hard_cons`` stay
    last-epoch numbers (comparable with runs scored before this was tracked);
    ``weighted_f1_best_epoch`` is what section 10 exports.
    """
    epoch_preds = sorted(run_dir.glob("predictions_dev_*.csv"))
    if not epoch_preds:
        raise FileNotFoundError(f"{key}: no predictions_dev_*.csv in {run_dir}")

    ref_path = track_paths(TRACK)[2]
    row = score_predictions(key, model_key, model_id, freeze, seed,
                            epoch_preds[-1], ref_path, run_dir, elapsed)

    history = read_history(run_dir)
    if not history:
        return row

    best_epoch = best_epoch_of(history)
    row["best_epoch"] = best_epoch
    # True when the best epoch is the last one: the run was still improving and
    # the epoch budget, not overfitting, decided where training stopped.
    row["best_is_last"] = best_epoch == len(epoch_preds)
    row["dev_f1_best"] = float(max(h["dev_f1_weighted"] for h in history))
    row["dev_f1_last"] = float(history[-1]["dev_f1_weighted"])
    row["train_loss_last"] = float(history[-1]["train_loss"])

    if not 1 <= best_epoch <= len(epoch_preds):
        print(f"warning: {key}: best epoch {best_epoch} has no prediction file "
              f"({len(epoch_preds)} found) - keeping the last-epoch score")
        return row

    best_pred = epoch_preds[best_epoch - 1]
    row["best_epoch_predictions"] = str(best_pred)
    if best_pred == epoch_preds[-1]:
        row["weighted_f1_best_epoch"] = row["weighted_f1"]
        row["soft_cons_best_epoch"] = row["soft_cons"]
        row["hard_cons_best_epoch"] = row["hard_cons"]
    else:
        scored = score_predictions(key, model_key, model_id, freeze, seed,
                                   best_pred, ref_path, run_dir)
        row["weighted_f1_best_epoch"] = scored["weighted_f1"]
        row["soft_cons_best_epoch"] = scored["soft_cons"]
        row["hard_cons_best_epoch"] = scored["hard_cons"]
    return row


def run_experiment(model_key, model_id, freeze, seed, *, overwrite=OVERWRITE):
    key = run_key(model_key, freeze, seed)
    run_dir = RESULTS_DIR / key
    log_path = LOG_DIR / f"{key}.log"

    if run_dir.exists() and not overwrite and sorted(run_dir.glob("predictions_dev_*.csv")):
        print(f"[skip] {key} -> scoring existing predictions")
        return score_run(key, model_key, model_id, freeze, seed, run_dir)

    if overwrite and run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    save_path = run_dir / "model" if SAVE_MODELS else None
    cmd = build_command(model_id, seed, run_dir, freeze, save_path=save_path)
    print(f"[run ] {key}")
    started = time.time()
    returncode = run_logged_command(cmd, STARTER_KIT, log_path)
    elapsed = time.time() - started

    if returncode != 0:
        print(f"[fail] {key}: exit {returncode}")
        print(f"---- last 30 lines of {log_path.name} ----")
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]
        for line in tail:
            print("    " + line)
        print("---- end of log ----")
        return None

    n_epochs = len(list(run_dir.glob("predictions_dev_*.csv")))
    print(f"[ok  ] {key} in {elapsed / 60:.1f} min -> {n_epochs} epoch prediction files")
    return score_run(key, model_key, model_id, freeze, seed, run_dir, elapsed)


In [ ]:
if SUMMARY_CSV.exists() and not OVERWRITE:
    results = pd.read_csv(SUMMARY_CSV).to_dict("records")
    done = {(r["model"], int(r["freeze_layers"]), int(r["seed"]),
             bool(r.get("symmetric", False)), str(r.get("twin_prior", "off")))
            for r in results}
else:
    results, done = [], set()

print(f"{len(done)}/{TOTAL} run(s) already scored\n")

for seed in SEEDS:
    if (MODEL_KEY, FREEZE_BOTTOM_LAYERS, seed, SYMMETRIC, TWIN_PRIOR) in done:
        print(f"[skip] {run_key(MODEL_KEY, FREEZE_BOTTOM_LAYERS, seed)}")
        continue
    try:
        row = run_experiment(MODEL_KEY, MODEL_ID, FREEZE_BOTTOM_LAYERS, seed)
    except Exception:
        traceback.print_exc()
        row = None
    if row is not None:
        results.append(row)
        done.add((MODEL_KEY, FREEZE_BOTTOM_LAYERS, seed, SYMMETRIC, TWIN_PRIOR))
        pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)
        gain = row.get("weighted_f1_best_epoch")
        gain = 0.0 if gain is None else gain - row["weighted_f1"]
        print(
            f"     -> weighted_f1={row['weighted_f1']:.4f}  "
            f"soft_cons={row['soft_cons']:.4f}  hard_cons={row['hard_cons']:.4f}  "
            f"trainable={row['trainable_pct']}%  "
            f"best_epoch={row.get('best_epoch')} (+{gain:.4f} vs last)"
        )

df = pd.read_csv(SUMMARY_CSV)
print(f"\nFinished: {len(df)} run(s) in {SUMMARY_CSV.name}.")


## 4. Scores

In [ ]:
# One row per run, ranked by the score the ZIP exports (best dev epoch).
show = df.copy()
show["shipped_f1"] = (show["weighted_f1_best_epoch"] if "weighted_f1_best_epoch" in show.columns
                      else show["weighted_f1"]).fillna(show["weighted_f1"])
cols = ["run", "seed", "freeze_layers", "trainable_pct", "shipped_f1",
        "weighted_f1", "soft_cons", "hard_cons", "best_epoch", "minutes"]
display(show.sort_values("shipped_f1", ascending=False)[[c for c in cols if c in show.columns]])

print("\nmean over seeds:")
display(show[["shipped_f1", "weighted_f1", "soft_cons", "hard_cons"]].mean().round(4).to_frame("mean"))
print("spread over seeds (max - min):")
display((show[["shipped_f1", "soft_cons", "hard_cons"]].max()
         - show[["shipped_f1", "soft_cons", "hard_cons"]].min()).round(4).to_frame("spread"))
print("\nA best_epoch well before the last epoch means the run overfits and the "
      "best-epoch export is what protects the submission.")


## 5. Error analysis - what does it actually get wrong?

In [ ]:
# Failure analysis: which dev rows does the run the ZIP would export actually miss?
# Everything below reads files already on disk - no training, no model, no hypothesis.
from evaluation_functions import ALL_LABELS

REFERENCE = track_paths(TRACK)[2]
LABELS = list(ALL_LABELS)
NEG = "NEGATIVE_OTHER"

# Pick the run the ZIP builder would: current inference settings, ranked by the
# shipped score (best dev epoch) with hard consistency as the tie-break.
pool = df.copy()
if "twin_prior" in pool.columns:
    matched = pool[pool["twin_prior"].fillna("off") == TWIN_PRIOR]
    if matched.empty:
        print(f"note: no run with twin_prior={TWIN_PRIOR!r} yet - "
              "analysing the best run with any settings")
    else:
        pool = matched
pool["_rank"] = (pool["weighted_f1_best_epoch"] if "weighted_f1_best_epoch" in pool.columns
                 else pool["weighted_f1"]).fillna(pool["weighted_f1"])
probe = pool.sort_values(["_rank", "hard_cons"], ascending=False).iloc[0]

probe_key = run_key(probe["model"], int(probe["freeze_layers"]), int(probe["seed"]))
probe_dir = RESULTS_DIR / probe_key
epoch_preds = sorted(probe_dir.glob("predictions_dev_*.csv"))
if not epoch_preds:
    raise FileNotFoundError(f"{probe_key}: no predictions_dev_*.csv in {probe_dir}")
best_epoch = (int(probe["best_epoch"]) if pd.notna(probe.get("best_epoch", float("nan")))
              else len(epoch_preds))
pred_path = epoch_preds[best_epoch - 1]

ref = pd.read_csv(REFERENCE, dtype=str)
pred = pd.read_csv(pred_path, dtype=str)
merged = ref.merge(pred, on="instance_id", how="left", suffixes=("_gold", "_pred"))
if merged["label_pred"].isna().any():
    raise ValueError(f"{pred_path.name}: {int(merged['label_pred'].isna().sum())} dev rows "
                     "have no prediction")

gold, got = merged["label_gold"], merged["label_pred"]
total = len(merged)

print(f"run   : {probe_key}")
print(f"epoch : {best_epoch} of {len(epoch_preds)}  ({pred_path.name})")
print(f"score : weighted F1 {probe['_rank']:.4f} at this epoch")
print(f"flags : twin_prior={probe.get('twin_prior', 'off')} "
      f"symmetric={bool(probe.get('symmetric', False))}")

# ---- per-label table: support share x F1 is what each class contributes ----
rows = []
for label in LABELS:
    tp = int(((gold == label) & (got == label)).sum())
    support = int((gold == label).sum())
    predicted = int((got == label).sum())
    precision = tp / predicted if predicted else 0.0
    recall = tp / support if support else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows.append({
        "label": label, "support": support, "predicted": predicted,
        "tp": tp, "fp": predicted - tp, "fn": support - tp,
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(f1, 4), "share": round(support / total, 4),
        "contribution": round(support / total * f1, 4),
    })
per_class = pd.DataFrame(rows).set_index("label")
print("\nPer-label breakdown (contribution = support share x F1):")
display(per_class)
print(f"sum of contributions = {per_class['contribution'].sum():.4f} (recomputed weighted F1)")

# ---- the NEG question, measured -------------------------------------------------
neg = per_class.loc[NEG]
rev_as_neg = int(((gold != NEG) & (got == NEG)).sum())
print("\nNEG, measured on this run:")
print(f"  gold NEG rows             : {int(neg['support'])}")
print(f"  NEG recall                : {int(neg['tp'])} / {int(neg['support'])} "
      f"= {neg['recall']:.3f}")
print(f"  NEG precision             : {neg['precision']:.3f} "
      f"({int(neg['fp'])} false positives)")
print(f"  reversible rows called NEG: {rev_as_neg}")
print(f"  NEG share of the score    : {neg['share']:.4f}")
print(f"  achieved from NEG         : {neg['contribution']:.4f}")
print(f"  still on the table        : {neg['share'] - neg['contribution']:.4f}")
print(f"  never predicting NEG caps : {(total - neg['support']) / total:.4f}")

# ---- where the errors are --------------------------------------------------------
errors = merged[merged["label_gold"] != merged["label_pred"]].copy()
errors["pattern"] = errors["label_gold"] + " -> " + errors["label_pred"]
print(f"\n{len(errors)} / {total} rows wrong ({len(errors) / total:.1%})")
display(errors["pattern"].value_counts().rename("rows").to_frame())

out = probe_dir / f"failures_epoch{best_epoch}.csv"
errors[["instance_id", "pair_id", "text1", "text2", "label_gold", "label_pred"]].to_csv(
    out, index=False)
print(f"all errors written to: {out}")

if errors.empty:
    print("\nNo dev errors: this run gets every row right.")
else:
    top = errors["pattern"].value_counts().index[0]
    print(f"\nLargest pattern: {top}")
    display(errors[errors["pattern"] == top].head(8)[
        ["instance_id", "text1", "text2", "label_gold", "label_pred"]])


## 6. Optional: the same config with the prior

The recipe above is the honest baseline (`TWIN_PRIOR = "off"`). To measure what
the `pair_id` exploit is worth on that same recipe, set `TWIN_PRIOR = "full"` in
section 2, re-run section 3, then run this cell with the prior set back to `"off"`.
The two run keys differ only in the `_prior-full` suffix.


In [ ]:
REFERENCE = track_paths(TRACK)[2]


def no_prior_key(model_key: str, freeze: int, seed: int) -> str:
    slug = model_key.lower().replace(".", "")
    return f"{slug}_track{TRACK}_f{freeze}_seed{seed}{'_sym' if SYMMETRIC else ''}"


def prior_key(model_key: str, freeze: int, seed: int) -> str:
    # Built explicitly: run_key() follows the *current* TWIN_PRIOR, so with the
    # default "off" it would resolve to the no-prior directory and compare a run
    # against itself.
    return no_prior_key(model_key, freeze, seed) + "_prior-full"


rows = []
for seed in SEEDS:
    with_prior = RESULTS_DIR / prior_key(MODEL_KEY, FREEZE_BOTTOM_LAYERS, seed)
    without = RESULTS_DIR / no_prior_key(MODEL_KEY, FREEZE_BOTTOM_LAYERS, seed)
    a = sorted(with_prior.glob("predictions_dev_*.csv"))
    b = sorted(without.glob("predictions_dev_*.csv"))
    if not a or not b:
        print(f"seed {seed}: no-prior run not found at {without.name} - skipped")
        continue
    # Training is identical up to the prior (the prior only touches evaluation),
    # but early stopping sees a different dev F1 and can stop at a different
    # epoch. Score both at the same epoch index so only the prior differs.
    epoch = min(len(a), len(b))
    scored = {}
    for tag, run_dir, preds in (("prior", with_prior, a), ("no_prior", without, b)):
        scored[tag] = score_predictions(run_dir.name, MODEL_KEY, MODEL_ID,
                                        FREEZE_BOTTOM_LAYERS, seed, preds[epoch - 1],
                                        REFERENCE, run_dir)
    rows.append({
        "seed": seed, "epoch": epoch,
        "no_prior_f1": scored["no_prior"]["weighted_f1"],
        "prior_f1": scored["prior"]["weighted_f1"],
        "delta_f1": scored["prior"]["weighted_f1"] - scored["no_prior"]["weighted_f1"],
        "no_prior_soft": scored["no_prior"]["soft_cons"],
        "prior_soft": scored["prior"]["soft_cons"],
        "no_prior_hard": scored["no_prior"]["hard_cons"],
        "prior_hard": scored["prior"]["hard_cons"],
    })

if rows:
    display(pd.DataFrame(rows).round(4))
    print(f"Both sides are scored at epoch {rows[0]['epoch']} (the last epoch both runs "
          "reached), so the only difference is the prior. The ZIP exports the best "
          "epoch instead, which need not be this one.")
else:
    print("Nothing to compare yet. Train the same config with TWIN_PRIOR='full' first.")


## 7. Ensemble across seeds

Each seed is an independent draw from the same recipe, so averaging their probabilities
before the argmax is the cheapest reliable gain available here - no extra training.
Members are chosen with the same filters as the ZIP (current `TWIN_PRIOR`, `SYMMETRIC`,
model, freeze depth), and each contributes its **own** best epoch rather than a shared
epoch index.

Every member is validated before averaging: `argmax(probabilities)` has to reproduce that
run's saved `predictions_dev_*.csv` exactly. That both catches a stale file and proves the
class order used below is the order the trainer actually used.

If the ensemble scores higher, the ZIP section ships it instead of the single best run.


In [ ]:
import numpy as np

ENSEMBLE_REF = track_paths(TRACK)[2]

# The trainer encodes labels with sklearn's LabelEncoder, which sorts the classes.
CLASS_ORDER = ["BACKWARD_ENTAILMENT", "EQUIVALENCE",
               "FORWARD_ENTAILMENT", "NEGATIVE_OTHER"]

pool = df.copy()
if "twin_prior" in pool.columns:
    pool = pool[pool["twin_prior"].fillna("off") == TWIN_PRIOR]
if "train_scope" in pool.columns:
    pool = pool[pool["train_scope"].fillna("single") == TRAIN_SCOPE]
if "symmetric" in pool.columns:
    pool = pool[pool["symmetric"].fillna(False).astype(bool) == SYMMETRIC]
pool = pool[(pool["model"] == MODEL_KEY)
            & (pool["freeze_layers"].astype(int) == FREEZE_BOTTOM_LAYERS)]

members, matrices, ids = [], [], None
for _, row in pool.sort_values("seed").iterrows():
    seed = int(row["seed"])
    key = run_key(str(row["model"]), int(row["freeze_layers"]), seed)
    run_dir = RESULTS_DIR / key
    prob_files = sorted(run_dir.glob("probabilities_dev_*.npy"))
    pred_files = sorted(run_dir.glob("predictions_dev_*.csv"))
    if not prob_files:
        print(f"[skip] {key}: no probabilities_dev_*.npy (trained without SAVE_PROBS)")
        continue
    best = row.get("best_epoch")
    epoch = int(best) if pd.notna(best) else len(prob_files)
    prob = np.load(prob_files[epoch - 1])
    saved = pd.read_csv(pred_files[epoch - 1], dtype=str)
    if ids is None:
        ids = saved["instance_id"].to_numpy()
    elif not (ids == saved["instance_id"].to_numpy()).all():
        raise ValueError(f"{key}: instance_id order differs from the first member")
    decoded = np.array(CLASS_ORDER)[prob.argmax(axis=1)]
    if not (decoded == saved["label"].to_numpy()).all():
        raise ValueError(f"{key}: argmax(probabilities) does not reproduce "
                         f"{pred_files[epoch - 1].name} - class order or epoch is wrong")
    members.append(f"seed{seed}@e{epoch}")
    matrices.append(prob)

SHIP_ENSEMBLE = False
ENSEMBLE_SIZE = 0
if not matrices:
    print("No probability files yet - retrain section 3 with SAVE_PROBS=True.")
else:
    ENSEMBLE_SIZE = len(matrices)
    mean = np.mean(np.stack(matrices), axis=0)
    ENSEMBLE_CSV = RESULTS_DIR / "predictions_dev_ensemble.csv"
    pd.DataFrame({
        "instance_id": ids,
        "label": np.array(CLASS_ORDER)[mean.argmax(axis=1)],
    }).to_csv(ENSEMBLE_CSV, index=False, lineterminator="\n")

    report = score_files(reference_path=ENSEMBLE_REF, prediction_path=ENSEMBLE_CSV,
                         output_dir=RESULTS_DIR / "ensemble_scores")
    rank_col = ("weighted_f1_best_epoch" if "weighted_f1_best_epoch" in pool.columns
                else "weighted_f1")
    single = float(pool[rank_col].fillna(pool["weighted_f1"]).max())

    print("members :", ", ".join(members))
    print(f"single  : {single:.4f}  (best seed, its best epoch)")
    print(f"ensemble: {report.weighted_f1:.4f}  (soft {report.soft_cons:.4f}, "
          f"hard {report.hard_cons:.4f})")
    print(f"delta   : {report.weighted_f1 - single:+.4f}")
    SHIP_ENSEMBLE = report.weighted_f1 > single
    print("-> the ZIP section will ship the "
          + ("ENSEMBLE" if SHIP_ENSEMBLE else "single best run"))


## 8. Submission ZIP (dev dry run)

In [ ]:
import hashlib
import zipfile

from evaluation_functions import ALL_LABELS
from evaluation_functions.io import read_prediction_file

# Section 7 sets these: ship the ensemble when it beat the best single seed.
SHIP_ENSEMBLE = bool(globals().get("SHIP_ENSEMBLE", False))
ENSEMBLE_SIZE = int(globals().get("ENSEMBLE_SIZE", 0))
ENSEMBLE_CSV = RESULTS_DIR / "predictions_dev_ensemble.csv"
if SHIP_ENSEMBLE and not ENSEMBLE_CSV.exists():
    raise FileNotFoundError(f"{ENSEMBLE_CSV} is missing - run section 7 first.")

# ---- choose what goes into the ZIP ---------------------------------------
# Only runs with the current inference settings: a prior run and a non-prior run
# of the same config differ in what they can predict, so mixing them would pick
# the winner of a comparison that does not hold.
if "twin_prior" in df.columns:
    df = df[df["twin_prior"].fillna("off") == TWIN_PRIOR].copy()
if "train_scope" in df.columns:
    df = df[df["train_scope"].fillna("single") == TRAIN_SCOPE].copy()

# Rank by the score the ZIP actually ships (best dev epoch). hard_cons breaks ties.
rank_col = "weighted_f1_best_epoch" if "weighted_f1_best_epoch" in df.columns else "weighted_f1"
rank = df.copy()
rank[rank_col] = rank[rank_col].fillna(rank["weighted_f1"])
best_row = rank.sort_values([rank_col, "hard_cons"], ascending=False).iloc[0]
# "best" = the epoch with the highest dev F1 (from training_history.json).
# Use -1 for the last epoch, 0 for the first, 2 for the third ...
EXPORT_EPOCH = "best"

# track -> (model_key, freeze_bottom_layers, seed). Add more tracks here later.
TRACK_RUNS = {
    1: (best_row["model"], int(best_row["freeze_layers"]), int(best_row["seed"])),
}

STAGE = RESULTS_DIR / "submission_dev"
shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir(parents=True, exist_ok=True)
zip_path = RESULTS_DIR / "dico_nli_dev_submission.zip"
zip_path.unlink(missing_ok=True)

manifest = []
for track, (model_key, freeze, seed) in TRACK_RUNS.items():
    key = run_key(model_key, freeze, seed)
    if SHIP_ENSEMBLE:
        key = f"ensemble_{ENSEMBLE_SIZE}_seeds"
        source = ENSEMBLE_CSV
    else:
        epoch_preds = sorted((RESULTS_DIR / key).glob("predictions_dev_*.csv"))
        if not epoch_preds:
            raise FileNotFoundError(f"{key}: no predictions_dev_*.csv - run section 3 first.")
        if EXPORT_EPOCH == "best":
            chosen = best_row.get("best_epoch")
            if chosen is None or pd.isna(chosen):
                print(f"warning: {key} has no best_epoch recorded - exporting the last epoch")
                source = epoch_preds[-1]
            else:
                source = epoch_preds[int(chosen) - 1]
        else:
            source = epoch_preds[EXPORT_EPOCH]

    template = pd.read_csv(
        DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_submission_template.csv", dtype=str
    )
    preds = pd.read_csv(source, dtype=str)

    if not preds["instance_id"].is_unique:
        raise ValueError(f"track {track}: duplicated instance_id in {source.name}")
    missing = set(template["instance_id"]) - set(preds["instance_id"])
    unknown = set(preds["instance_id"]) - set(template["instance_id"])
    if missing or unknown:
        raise ValueError(f"track {track}: {len(missing)} missing / {len(unknown)} unknown instance_ids")

    # Reorder to the template. CodaBench does not require it, but it makes the
    # upload reproducible and diffable against the template.
    preds = preds.set_index("instance_id").loc[template["instance_id"]].reset_index()

    csv_name = f"track{track}_predictions.csv"          # mandated filename
    dst = STAGE / csv_name
    preds.to_csv(dst, index=False, lineterminator="\n")  # header: instance_id,label

    manifest.append({
        "track": track,
        "file": csv_name,
        "run": key,
        "epoch_selection": ("mean of best epochs" if SHIP_ENSEMBLE else str(EXPORT_EPOCH)),
        "source": source.name,
        "rows": len(preds),
        "labels": preds["label"].nunique(),
        "sha256": hashlib.sha256(dst.read_bytes()).hexdigest()[:16],
    })

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in manifest:
        archive.write(STAGE / item["file"], arcname=item["file"])   # arcname has no directories

print(f"Wrote {zip_path} ({zip_path.stat().st_size:,} bytes)")
display(pd.DataFrame(manifest))


In [ ]:
# Re-open the ZIP and check it the way CodaBench does.
ACCEPTED = re.compile(r"^track[1-4]_predictions\.csv$")

with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    print("ZIP entries:", names)

    assert names, "ZIP is empty"
    assert all("/" not in n and "\\" not in n for n in names), "files must sit at the ZIP root"
    assert all(ACCEPTED.match(n) for n in names), f"unexpected file(s): {names}"
    assert not any(n.startswith("__MACOSX") or n.endswith(".DS_Store") for n in names)

    for name in names:
        track = int(name[5])
        staged = STAGE / name
        assert archive.read(name) == staged.read_bytes(), f"{name}: ZIP content differs from staged file"

        records = read_prediction_file(staged)          # official header + label validation
        ids = [record.instance_id for record in records]
        assert len(ids) == len(set(ids)), f"{name}: duplicated instance_id"

        template = pd.read_csv(
            DATA_ROOT / "dev" / f"dico_nli_dev_track{track}_submission_template.csv", dtype=str
        )
        assert set(ids) == set(template["instance_id"]), f"{name}: instance_id mismatch"
        assert {record.label for record in records} <= set(ALL_LABELS), f"{name}: invalid label"

        first_lines = staged.read_text(encoding="utf-8").splitlines()[:2]
        print(f"\n{name}: {len(records)} predictions")
        print(f"  header : {first_lines[0]}")
        print(f"  first  : {first_lines[1]}")
        counts = pd.Series([record.label for record in records]).value_counts()
        print(f"  labels : {counts.to_dict()}")

print(f"\nOK - upload this archive to CodaBench: {zip_path}")
print("Single-track ZIP: track 1 is scored; tracks 2-4 and macro4_* show -1.0.")
